# Final holdout evaluation (April 2016 - September 2026)

Runs the **frozen** strategy in `config/frozen_strategy.json` exactly once. `authorize_final` checks the SHA-256 lock, refuses a different config, and appends this run to `results/final_runs.jsonl`. Small result tables and the chart go to `results/` (committed); per-trade tables to `outputs/final/` (ignored).

In [ ]:
acknowledge_reselection = False
note = "final evaluation"

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'quantlab').is_dir())
sys.path.insert(0, str(root))
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from quantlab import protocol
from quantlab.experiment import run_final
cfg = json.loads(protocol.FROZEN_CONFIG.read_text())
auth = protocol.authorize_final(cfg, acknowledge_reselection=acknowledge_reselection, note=note)
print(auth.config_sha256, 'prior final runs:', auth.prior_final_runs)

In [ ]:
fin = run_final(cfg, auth)
print(json.dumps(fin['summary'], indent=1, default=str))

## Required comparison: strategy vs $1,000,000 in ^SP500TR at the first test close

In [ ]:
display(fin['performance'])
display(fin['relative'].round(3))
display(Image(filename=str(root / 'results' / 'cumulative_return.png')))

## Accounting and timing audit of the final run

In [ ]:
display(fin['audit'])
assert fin['audit'].passed.all()

## Sensitivity: data repairs, frozen-lot write-off, and costs

In [ ]:
display(fin['sensitivity'][['final_value','CAGR','max_drawdown','Sharpe']])

## Prediction quality on completed test labels

In [ ]:
display(fin['prediction'])
display(fin['prediction_by_year'].round(4))

## Portfolio diagnostics and data incidents touching held lots

In [ ]:
display(fin['diagnostics'].round(4))
display(fin['touched'].groupby('kind').size())
display(fin['touched'].sort_values('date').head(30))